In [ ]:
# =========================
# Imports für alle Modelle
# =========================
import pandas as pd
import numpy as np

# Scikit-Learn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

# Matplotlib (für Visualisierung, falls benötigt)
import matplotlib.pyplot as plt

# Keras/Tensorflow für das neuronale Netz
from scikeras.wrappers import KerasRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD

# =========================
# Ende Imports
# =========================

# Information
This notebook defines the necessary preprocessing steps to prepare the player_stats dataset for model training. A subset of these steps will later be exported into a standalone Python module and integrated into the data loading and preprocessing pipeline used for inference. These reusable steps are explicitly labeled throughout the notebook.

The remaining steps are specific to the data integration process and are required to merge data from Transfermarkt (TM) with data from Sofascore. This integration is necessary to link player_stats with the corresponding player ratings (grades) and is not part of the final preprocessing pipeline used during model deployment.

In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/transform/pro/stats_with_rating.csv")

In [6]:
cols = ["player_id", "match_id", "club_id", "player_name", "date",  "home_club_id", "away_club_id", "home_goals", "away_goals"]
df = df.drop(columns = cols, errors="ignore")

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = ["goals", "assists", "minutes", "team_goals", "team_conceded"]
cat_cols = ["position", "result"]
bool_cols = ["yellow", "red", "start_eleven"]

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("bool", "passthrough", bool_cols),
    ]
)

X = df.drop(columns=["rating"])
y = df["rating"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("bool", "passthrough", bool_cols),
    ]
)


# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LinearRegression())
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 0.3159168278823548
RMSE: 0.43304838305377036


In [12]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split, GridSearchCV
import pandas as pd
import matplotlib.pyplot as plt


preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("bool", "passthrough", bool_cols),
    ]
)

# -------------------
# Data Split
# -------------------
X = df.drop(columns=["rating"])
y = df["rating"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1))
])

param_grid = {
    "model__n_estimators": [100, 400, 1000],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)


best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)

print("Best Params:", grid.best_params_)
print("MAE:", mae)
print("RMSE:", rmse)


Fitting 5 folds for each of 48 candidates, totalling 240 fits
Best Params: {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 400}
MAE: 0.2508098969567457
RMSE: 0.3603172916638304


In [9]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, Lasso

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("bool", "passthrough", bool_cols),
    ]
)

# Ridge
pipeline_ridge = Pipeline([
    ("preprocessing", preprocessor),
    ("model", Ridge(alpha=1.0))
])

pipeline_ridge.fit(X_train, y_train)
y_pred_ridge = pipeline_ridge.predict(X_test)

mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
rmse_ridge = root_mean_squared_error(y_test, y_pred_ridge)

print("Ridge MAE:", mae_ridge)
print("Ridge RMSE:", rmse_ridge)

# Lasso
pipeline_lasso = Pipeline([
    ("preprocessing", preprocessor),
    ("model", Lasso(alpha=0.1))
])

pipeline_lasso.fit(X_train, y_train)
y_pred_lasso = pipeline_lasso.predict(X_test)

mae_lasso = mean_absolute_error(y_test, y_pred_lasso)
rmse_lasso = root_mean_squared_error(y_test, y_pred_lasso)

print("Lasso MAE:", mae_lasso)
print("Lasso RMSE:", rmse_lasso)

Ridge MAE: 0.3158885213296492
Ridge RMSE: 0.4330550555954091
Lasso MAE: 0.36138950518241914
Lasso RMSE: 0.4966606087655508


In [13]:
# =========================
# Imports
# =========================
from scikeras.wrappers import KerasRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

# =========================
# Model Builder (mit meta 🔥)
# =========================
def build_model(meta, activation="relu", alpha=0.001, dropout_rate=0.2, momentum=0.0):
    model = Sequential()

    # Input automatisch von scikeras
    model.add(Dense(128, activation=activation, input_shape=(meta["n_features_in_"],)))
    model.add(Dropout(dropout_rate))
    
    model.add(Dense(64, activation=activation))
    model.add(Dropout(dropout_rate))
    
    model.add(Dense(32, activation=activation))

    model.add(Dense(1))

    optimizer = SGD(learning_rate=alpha, momentum=momentum)

    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=["mae"]
    )

    return model

# =========================
# Wrapper
# =========================
nn = KerasRegressor(
    model=build_model,
    epochs=50,
    batch_size=32,
    verbose=0
)

# =========================
# Pipeline
# =========================
pipeline_nn = Pipeline([
    ("preprocessing", preprocessor),
    ("model", nn)
])

# =========================
# Grid
# =========================
param_grid = {
    "model__model__activation": ["relu", "tanh"],
    "model__model__alpha": [0.001, 0.01],
    "model__model__dropout_rate": [0.2, 0.3],
    "model__model__momentum": [0.0, 0.9]
}

# =========================
# GridSearch
# =========================
grid = GridSearchCV(
    pipeline_nn,
    param_grid,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)

# =========================
# Evaluation
# =========================
y_pred_nn = grid.predict(X_test)

mae_nn = mean_absolute_error(y_test, y_pred_nn)
rmse_nn = root_mean_squared_error(y_test, y_pred_nn)

print("NN MAE:", mae_nn)
print("NN RMSE:", rmse_nn)

c:\Users\fabia\.conda\envs\kerasenv\lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Best Params: {'model__model__activation': 'relu', 'model__model__alpha': 0.01, 'model__model__dropout_rate': 0.2, 'model__model__momentum': 0.0}
NN MAE: 0.31718237510163416
NN RMSE: 0.43840630560665855
